In [ ]:
import os
import glob
import random
import itertools
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
import matplotlib.pyplot as plt

# =================================================================
# 1. CẤU HÌNH HỆ THỐNG
# =================================================================
# Giả sử thư mục gốc chứa 4 folder là: /kaggle/input/horse2zebra-dataset/
ROOT_DATA = "/kaggle/input/datasets/balraj98/horse2zebra-dataset" 
SAVE_DIR = "/kaggle/working/checkpoints"
os.makedirs(SAVE_DIR, exist_ok=True)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
IMG_SIZE = 256
BATCH_SIZE = 1
LR = 0.0002
EPOCHS = 100
LAMBDA_CYCLE = 10.0
LAMBDA_ID = 5.0

# =================================================================
# 2. KIẾN TRÚC MẠNG (GENERATOR & DISCRIMINATOR)
# =================================================================

class ResNetBlock(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.block = nn.Sequential(
            nn.ReflectionPad2d(1),
            nn.Conv2d(dim, dim, 3),
            nn.InstanceNorm2d(dim),
            nn.ReLU(inplace=True),
            nn.ReflectionPad2d(1),
            nn.Conv2d(dim, dim, 3),
            nn.InstanceNorm2d(dim)
        )
    def forward(self, x):
        return x + self.block(x)

class Generator(nn.Module):
    def __init__(self, n_res=9):
        super().__init__()
        # Initial
        model = [nn.ReflectionPad2d(3), nn.Conv2d(3, 64, 7), nn.InstanceNorm2d(64), nn.ReLU(inplace=True)]
        # Downsampling
        curr_dim = 64
        for _ in range(2):
            model += [nn.Conv2d(curr_dim, curr_dim*2, 3, stride=2, padding=1),
                      nn.InstanceNorm2d(curr_dim*2), nn.ReLU(inplace=True)]
            curr_dim *= 2
        # Residual
        for _ in range(n_res):
            model += [ResNetBlock(curr_dim)]
        # Upsampling
        for _ in range(2):
            model += [nn.ConvTranspose2d(curr_dim, curr_dim//2, 3, stride=2, padding=1, output_padding=1),
                      nn.InstanceNorm2d(curr_dim//2), nn.ReLU(inplace=True)]
            curr_dim //= 2
        # Output
        model += [nn.ReflectionPad2d(3), nn.Conv2d(64, 3, 7), nn.Tanh()]
        self.model = nn.Sequential(*model)

    def forward(self, x): return self.model(x)

class Discriminator(nn.Module):
    def __init__(self):
        super().__init__()
        def d_layer(in_f, out_f, stride=2, norm=True):
            layers = [nn.Conv2d(in_f, out_f, 4, stride=stride, padding=1)]
            if norm: layers.append(nn.InstanceNorm2d(out_f))
            layers.append(nn.LeakyReLU(0.2, inplace=True))
            return layers
        
        self.model = nn.Sequential(
            *d_layer(3, 64, norm=False),
            *d_layer(64, 128),
            *d_layer(128, 256),
            *d_layer(256, 512, stride=1),
            nn.Conv2d(512, 1, 4, padding=1)
        )
    def forward(self, x): return self.model(x)

# =================================================================
# 3. DATASET (Xử lý đúng 4 folder: trainA, trainB, testA, testB)
# =================================================================

class CycleGANDataset(Dataset):
    def __init__(self, root, transform=None, mode='train'):
        self.transform = transform
        # Kết nối trực tiếp vào tên folder bạn đã cung cấp
        self.path_A = os.path.join(root, f"{mode}A")
        self.path_B = os.path.join(root, f"{mode}B")
        
        self.files_A = sorted(glob.glob(self.path_A + "/*.*"))
        self.files_B = sorted(glob.glob(self.path_B + "/*.*"))

    def __getitem__(self, index):
        img_A = Image.open(self.files_A[index % len(self.files_A)]).convert("RGB")
        # CycleGAN lấy ảnh B ngẫu nhiên (unaligned)
        img_B = Image.open(random.choice(self.files_B)).convert("RGB")
        
        if self.transform:
            img_A = self.transform(img_A)
            img_B = self.transform(img_B)
        return {"A": img_A, "B": img_B}

    def __len__(self):
        return max(len(self.files_A), len(self.files_B))

# =================================================================
# 4. IMAGE POOL (Ổn định Discriminator)
# =================================================================

class ImagePool:
    def __init__(self, max_size=50):
        self.max_size = max_size
        self.data = []
    def query(self, data):
        if self.max_size <= 0: return data
        return_data = []
        for element in data:
            element = torch.unsqueeze(element, 0)
            if len(self.data) < self.max_size:
                self.data.append(element)
                return_data.append(element)
            else:
                if random.uniform(0, 1) > 0.5:
                    idx = random.randint(0, self.max_size - 1)
                    return_data.append(self.data[idx].clone())
                    self.data[idx] = element
                else:
                    return_data.append(element)
        return torch.cat(return_data, 0)

# =================================================================
# 5. HUẤN LUYỆN (TRAINING LOOP)
# =================================================================

def train():
    # Khởi tạo mạng
    G_AB = Generator().to(DEVICE)
    G_BA = Generator().to(DEVICE)
    D_A = Discriminator().to(DEVICE)
    D_B = Discriminator().to(DEVICE)

    # Khởi tạo Optimizer
    opt_G = optim.Adam(itertools.chain(G_AB.parameters(), G_BA.parameters()), lr=LR, betas=(0.5, 0.999))
    opt_D_A = optim.Adam(D_A.parameters(), lr=LR, betas=(0.5, 0.999))
    opt_D_B = optim.Adam(D_B.parameters(), lr=LR, betas=(0.5, 0.999))

    # Loss
    criterion_GAN = nn.MSELoss()
    criterion_cycle = nn.L1Loss()
    criterion_id = nn.L1Loss()

    # Data Loader
    tf = transforms.Compose([
        transforms.Resize(int(IMG_SIZE * 1.12), Image.BICUBIC),
        transforms.RandomCrop(IMG_SIZE),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
    ])
    
    dataset = CycleGANDataset(ROOT_DATA, transform=tf, mode='train')
    dataloader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True)

    fake_A_pool = ImagePool()
    fake_B_pool = ImagePool()

    print("--- Bắt đầu huấn luyện CycleGAN ---")
    for epoch in range(EPOCHS):
        for i, batch in enumerate(dataloader):
            real_A = batch["A"].to(DEVICE)
            real_B = batch["B"].to(DEVICE)

            # --- 1. TRAIN GENERATORS ---
            opt_G.zero_grad()

            # Identity loss
            loss_id_A = criterion_id(G_BA(real_A), real_A) * LAMBDA_ID
            loss_id_B = criterion_id(G_AB(real_B), real_B) * LAMBDA_ID

            # GAN loss
            fake_B = G_AB(real_A)
            loss_GAN_AB = criterion_GAN(D_B(fake_B), torch.ones_like(D_B(fake_B)))
            fake_A = G_BA(real_B)
            loss_GAN_BA = criterion_GAN(D_A(fake_A), torch.ones_like(D_A(fake_A)))

            # Cycle loss
            rec_A = G_BA(fake_B)
            loss_cycle_A = criterion_cycle(rec_A, real_A) * LAMBDA_CYCLE
            rec_B = G_AB(fake_A)
            loss_cycle_B = criterion_cycle(rec_B, real_B) * LAMBDA_CYCLE

            # Total G
            loss_G = loss_GAN_AB + loss_GAN_BA + loss_cycle_A + loss_cycle_B + loss_id_A + loss_id_B
            loss_G.backward()
            opt_G.step()

            # --- 2. TRAIN DISCRIMINATORS ---
            # D_A
            opt_D_A.zero_grad()
            loss_real_A = criterion_GAN(D_A(real_A), torch.ones_like(D_A(real_A)))
            f_A_sample = fake_A_pool.query(fake_A)
            loss_fake_A = criterion_GAN(D_A(f_A_sample.detach()), torch.zeros_like(D_A(f_A_sample)))
            loss_D_A = (loss_real_A + loss_fake_A) / 2
            loss_D_A.backward()
            opt_D_A.step()

            # D_B
            opt_D_B.zero_grad()
            loss_real_B = criterion_GAN(D_B(real_B), torch.ones_like(D_B(real_B)))
            f_B_sample = fake_B_pool.query(fake_B)
            loss_fake_B = criterion_GAN(D_B(f_B_sample.detach()), torch.zeros_like(D_B(f_B_sample)))
            loss_D_B = (loss_real_B + loss_fake_B) / 2
            loss_D_B.backward()
            opt_D_B.step()

        print(f"Epoch {epoch} | Loss G: {loss_G.item():.4f} | Loss D: {(loss_D_A + loss_D_B).item():.4f}")
        
        if epoch % 10 == 0:
            torch.save(G_AB.state_dict(), f"{SAVE_DIR}/G_AB_v2.pth")

# Chạy main
if __name__ == "__main__":
    train()

--- Bắt đầu huấn luyện CycleGAN ---
